# Testing DATA integration through Tango

This notebook checks the real-hardware DATA flow:

1. Connect to Tango devices.
2. Configure the DATA Tango device.
3. Acquire a STEM image from ThermoMicroscope.
4. Use the returned saved file path to retrieve data through the DATA device.


### Quick Start Code Cell

In [ ]:
import subprocess
import os
import json
import time
from getpass import getpass

import tango
import numpy as np
import matplotlib.pyplot as plt


## 0. Ping Tango servers


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DB_HOST = "10.46.217.241" # "localhost" # 
DB_PORT = 9094
server_list = [("stage", "asyncroscopy.hardware.STAGE"),
               ("scan", "asyncroscopy.hardware.SCAN"),
               ("eds", "asyncroscopy.detectors.EDS"),
               ("camera", "asyncroscopy.detectors.CAMERA"),
               ("data", "asyncroscopy.software.DATA"),
               ("corrector", "asyncroscopy.hardware.CORRECTOR"),
               ("microscope", "asyncroscopy.ThermoMicroscope")]
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = os.path.dirname(os.getcwd())
os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"
env = {**os.environ, "TANGO_HOST": f"{DB_HOST}:{DB_PORT}"}
processes = {}


def wait_for_device(name, timeout=2, interval=1):
    print(f"  Waiting for {name}...", end="", flush=True)
    start = time.time()
    while time.time() - start < timeout:
        try:
            tango.DeviceProxy(name).ping()
            print(f" ✅ ready ({time.time()-start:.1f}s)")
            return True
        except Exception:
            print(".", end="", flush=True)
            time.sleep(interval)
    print(f" ❌ timed out after {timeout}s")
    return False

def check_processes(*names):
    for name in names:
        p = processes[name]
        print(f"\n─── {name} (PID {p.pid}) ───\n  Running: {p.poll() is None}")
        for label, fd in [("STDOUT", p.stdout), ("STDERR", p.stderr)]:
            try:
                print(f"  {label}: {fd.read1(4096).decode() or '(empty)'}")
            except Exception:
                print(f"  {label}: (no output yet)")



if not all(wait_for_device(f"asyncroscopy/{d}/default") for d in ["stage", "scan", "eds", "camera", "data"]):
    print("\n⚠️  Debug info:")
    check_processes("stage", "scan", "eds", "camera", "data")
    raise RuntimeError("One or more device servers failed.")


print("\n✅ All servers ready!")



## 1. Connect to devices

In [ ]:
db = tango.Database()
print("Devices registered in Tango DB:\n")
for device in db.get_device_name("*", "*"):
    print(device)


In [ ]:
SCAN_DEVICE = "asyncroscopy/scan/default"
MICROSCOPE_DEVICE = "asyncroscopy/microscope/default"
DATA_DEVICE = "asyncroscopy/data/default"

In [ ]:
scan = tango.DeviceProxy(SCAN_DEVICE)
microscope = tango.DeviceProxy(MICROSCOPE_DEVICE)


for proxy in (scan, microscope):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())


In [ ]:
data = tango.DeviceProxy(DATA_DEVICE)
data.set_timeout_millis(120_000)

TILED_HOST = "10.46.217.241"
TILED_PORT = 9091

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = "D:/microscopedata/ahoust17/20260520/"

data.set_api_key(getpass("Enter your Tiled API key: "))
print(json.dumps(json.loads(data.get_config()), indent=2))

## 3. Configure a small STEM acquisition

In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 128

print("dwell_time:", scan.dwell_time)
print("imsize    :", scan.imsize)


## 4. Acquire and inspect the saved path

In [ ]:
saved_path = microscope.get_scanned_image()

print("Saved file:", saved_path)

print("\nDATA device path check:")
try:
    print(json.dumps(json.loads(data.path_exists(saved_path)), indent=2))
except Exception as exc:
    print("Could not run data.path_exists:", exc)


## 5. Check recent files from the DATA device

In [ ]:
print("Tiled root entries:")
try:
    print(json.dumps(json.loads(data.list_root()), indent=2)[:4000])
except Exception as exc:
    print("Could not list Tiled root:", exc)

recent = json.loads(data.get_recent())
print("\nRecent files visible to the DATA Tango device process:")
print(json.dumps(recent, indent=2)[:4000])


## 6. Resolve the saved path through Tiled

In [ ]:
def get_data_with_retry(saved_path, timeout=30, interval=2):
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            return json.loads(data.get_data(saved_path))
        except Exception as exc:
            last_error = exc
            print("Tiled lookup not ready yet:", exc)
            time.sleep(interval)

    raise RuntimeError(f"Could not resolve data through Tiled after {timeout}s") from last_error

data = get_data_with_retry(saved_path)

if isinstance(data, dict):
    summary = {k: data[k] for k in data.keys() if k != "data"}
    print(json.dumps(summary, indent=2)[:4000])
else:
    print(type(data), str(data)[:1000])


## 7. Plot if Tiled returned an array

In [ ]:
if isinstance(data, dict) and data.get("type") == "ndarray":
    image = np.asarray(data["data"], dtype=np.dtype(data["dtype"]))
    image = image.reshape(data["shape"])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray", interpolation="none")
    ax.set_title("HAADF from Tiled")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Tiled did not return a direct ndarray. Inspect data and traverse the returned node/path as needed.")
